In [ ]:
!pip install pysam

In [ ]:
import pandas as pd
from tqdm import tqdm
import pysam

In [ ]:
b = 0

In [ ]:
inds_duos = {}

with open(f'/mnt/project/DNM/validation/blocks/duo_pairs_b{b}.txt') as F:
            
    for i, row in enumerate(F):
                
        row = row.strip()
        row = row.split(' ')
    
        child = str(row[0])
        parent = str(row[1])
            
        inds_duos[child] = parent

In [ ]:
inds_diffs = {}

sibs_dnms_df = pd.read_csv('/mnt/project/DNM/sibs/out/sibs_diffs_info_updated.txt', sep = '\t')

for _, row in sibs_dnms_df.iterrows():
    
    ind_idx = row['IID']
    ind = str(ind_idx.split('_')[0])
    
    chrom = int(row['CHROM'])
    pos = int(row['POS'])
    ref = row['ref']
    alt = row['alt']
    
    if ind not in inds_diffs:
        inds_diffs[ind] = set()
    inds_diffs[ind].add(f'{chrom}:{pos}:{ref}:{alt}')

In [ ]:
inds_path = {}

# paths = '/mnt/project/Bulk/DRAGEN WGS/Whole genome variant call files (GVCFs) (DRAGEN) [500k release]/'

for ind in inds_duos:

    if ind not in inds_diffs:
        continue
    
    # inds_path[ind] = f'{paths}{ind[:2]}/{ind}_24051_0_0.dragen.hard-filtered.gvcf.gz'
    inds_path[ind] = f'{ind}_24051_0_0.dragen.hard-filtered.gvcf.gz'

    # inds_path[inds_duos[ind]] = f'{paths}{inds_duos[ind][:2]}/{inds_duos[ind]}_24051_0_0.dragen.hard-filtered.gvcf.gz'
    inds_path[inds_duos[ind]] = f'{inds_duos[ind]}_24051_0_0.dragen.hard-filtered.gvcf.gz'

In [ ]:
inds_diffs_qc = {}
inds_diffs_errors = {}

for _, ind in tqdm(enumerate(inds_duos)):

    if ind not in inds_diffs:
        continue
    
    # Parent --------------------------------
    
    parent = inds_duos[ind]
    
    try:
        vcf = pysam.VariantFile(inds_path[parent])
    except:
        continue # failed to read gVCF

    parent_info = {}
    
    for diff_id in inds_diffs[ind]:
        
        chrom = int(diff_id.split(':')[0])
        pos = int(diff_id.split(':')[1])
        ref = diff_id.split(':')[2]
        alt = diff_id.split(':')[3]
            
        for F in vcf.fetch(f'chr{chrom}', pos-1, pos, reopen = True):
            
            for f in list(F.samples):
            
                dp = F.samples[f]['DP']
                gq = F.samples[f]['GQ']
                gt = F.samples[f]['GT']
                
                if None in gt:
                    continue
                
                if F.pos == pos:
                    
                    ref_ad = 0
                    alt_ad = 0
                    
                    if ref in F.alleles:
                        i = F.alleles.index(ref)
                        ref_ad = F.samples[f]['AD'][i]
                        
                    if alt in F.alleles:
                        i = F.alleles.index(alt)
                        alt_ad = F.samples[f]['AD'][i]
                        
                    alleles = (F.alleles[gt[0]], F.alleles[gt[1]])
                        
                else:
                    ref_ad = dp
                    alt_ad = 0
                    alleles = (ref, ref)
                
                parent_info[diff_id] = (ref_ad, alt_ad, dp, gq, alleles)
    
    # Individual --------------------------------
    
    try:
        vcf = pysam.VariantFile(inds_path[ind])
    except:
        continue # failed to read gVCF
    
    inds_diffs_qc[ind] = []
    inds_diffs_errors[ind] = []
    
    for diff_id in inds_diffs[ind]:

        if diff_id not in parent_info:
            continue
            
        chrom = int(diff_id.split(':')[0])
        pos = int(diff_id.split(':')[1])
        ref = diff_id.split(':')[2]
        alt = diff_id.split(':')[3]
            
        for F in vcf.fetch(f'chr{chrom}', pos-1, pos, reopen = True):
            
            for f in list(F.samples):
            
                dp = F.samples[f]['DP']
                gq = F.samples[f]['GQ']
                gt = F.samples[f]['GT']
                
                if None in gt:
                    continue
                
                if F.pos == pos:
                    
                    ref_ad = 0
                    alt_ad = 0
                    
                    if ref in F.alleles:
                        i = F.alleles.index(ref)
                        ref_ad = F.samples[f]['AD'][i]
                        
                    if alt in F.alleles:
                        i = F.alleles.index(alt)
                        alt_ad = F.samples[f]['AD'][i]
                        
                    alleles = (F.alleles[gt[0]], F.alleles[gt[1]])
                        
                else:
                    ref_ad = dp
                    alt_ad = 0
                    alleles = (ref, ref)
                    
                info = (diff_id, ref_ad, alt_ad, dp, gq, alleles,
                        parent_info[diff_id][0], parent_info[diff_id][1], parent_info[diff_id][2], parent_info[diff_id][3], parent_info[diff_id][4])
                        
                if ((ref in alleles and alt in alleles) and
                    (parent_info[diff_id][4][0] == parent_info[diff_id][4][1] and ref in parent_info[diff_id][4])):
                    inds_diffs_qc[ind].append(info)
                else:
                    inds_diffs_errors[ind].append(info)

In [ ]:
with open(f'duos_qc_b{b}.txt', 'w') as f:
    for ind in inds_diffs_qc:
        for (diff_id, ref_ad, alt_ad, dp, gq, alleles, parent_ref_ad, parent_alt_ad, parent_dp, parent_gq, parent_alleles) in inds_diffs_qc[ind]:
            f.write(f'{ind} {diff_id} {ref_ad} {alt_ad} {dp} {gq} {alleles} {parent_ref_ad} {parent_alt_ad} {parent_dp} {parent_gq} {parent_alleles}\n')

In [ ]:
with open(f'duos_errors_b{b}.txt', 'w') as f:
    for ind in inds_diffs_errors:
        for (diff_id, ref_ad, alt_ad, dp, gq, alleles, parent_ref_ad, parent_alt_ad, parent_dp, parent_gq, parent_alleles) in inds_diffs_errors[ind]:
            f.write(f'{ind} {diff_id} {ref_ad} {alt_ad} {dp} {gq} {alleles} {parent_ref_ad} {parent_alt_ad} {parent_dp} {parent_gq} {parent_alleles}\n')